# 02 — Expressibility Check: Pre-Flight Diagnostics

Before running the full pipeline, this notebook helps you determine:
1. **Is your configuration viable?** (model × topology × N × p × h-range)
2. **What is the valid operating regime?** (h_min frontier)
3. **What are the hardware resource costs?** (CX gates, circuit depth)

The expressibility frontier h_min(N, p) defines where the HVA ansatz
can no longer represent the ground state (ΔE/gap > 5%). Operating
below this boundary is futile — it's a physics limit, not an
optimization failure.

**Key findings from our atlas (500+ runs):**
- TFIM chain_1d p=1: h_min ≈ 2.4 + 0.007·N
- TFIM chain_1d p=2: h_min ≈ 1.6 + 0.005·N  
- TFIM chain_1d p≥3: h_min ≈ 1.4-1.6 (quasi-constant)
- Heisenberg: h_min > 3.5 for ALL p (HVA incompatible)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qmbp_simulation import make_lattice, HVACircuitBuilder
from qmbp_simulation.models.model_registry import get_model_spec, list_models

print("Available models:", list_models())

# Visualize all topologies side by side
from viz_helpers import draw_topology_comparison, draw_lattice, draw_circuit
fig = draw_topology_comparison(["chain_1d", "heavy_hex", "ladder", "square", "triangular"])
plt.show()

## 1. Expressibility Frontier — h_min(N, p)

The frontier defines the minimum transverse field where the pipeline is
guaranteed to work. Below this boundary, the HVA at depth p cannot
capture the entanglement of the ground state.

In [ ]:
# ── Expressibility boundary fits (from 500+ empirical runs) ────────────
# These are linear fits to h_min vs N for TFIM chain_1d.
# Source: results/HVA_EXPRESSIBILITY_ANALYSIS.md

def h_min_frontier(N: int, p: int, model: str = "tfim", topology: str = "chain_1d") -> float:
    """Estimate the expressibility boundary h_min for a given configuration.
    
    Returns the minimum h where ΔE/gap < 5% is achievable.
    Values below this are physically inaccessible with HVA at depth p.
    """
    if model in ("heisenberg", "xy", "kitaev"):
        return float('inf')  # HVA incompatible
    
    # Topology multipliers (relative to chain_1d)
    topo_mult = {
        "chain_1d": 1.0,
        "heavy_hex": 1.03,   # Nearly identical to chain
        "ladder": 1.53,
        "kagome": 1.37,
        "square": 1.54,
        "triangular": 2.02,
    }
    mult = topo_mult.get(topology, 1.5)
    
    # Linear fits for chain_1d (from empirical data N=10-100)
    if p == 1:
        h_chain = 2.355 + 0.0073 * N
    elif p == 2:
        h_chain = 1.574 + 0.005 * N
    elif p >= 3:
        h_chain = 1.4 + 0.002 * N  # Quasi-constant
    else:
        h_chain = 3.0  # Conservative fallback
    
    return h_chain * mult


# Visualize the frontier
N_range = np.arange(4, 102, 2)

fig, ax = plt.subplots(figsize=(10, 6))
for p in [1, 2, 3, 4]:
    h_mins = [h_min_frontier(n, p) for n in N_range]
    ax.plot(N_range, h_mins, '-', lw=2, label=f'p={p}')

ax.axhline(1.0, color='red', ls='--', lw=1.5, label='h_c = 1.0 (QPT)')
ax.fill_between(N_range, 0, 1.0, alpha=0.1, color='red', label='Ordered phase (h < h_c)')
ax.set_xlabel('System size N (qubits)', fontsize=12)
ax.set_ylabel('h_min (expressibility boundary)', fontsize=12)
ax.set_title('HVA Expressibility Frontier — TFIM chain_1d', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(4, 100)
ax.set_ylim(0.5, 4.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Pre-Flight Viability Check

Enter your desired configuration and get a viability report.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CONFIGURE YOUR EXPERIMENT HERE
# ══════════════════════════════════════════════════════════════════════════
CONFIG = {
    "model": "tfim_longitudinal",
    "topology": "heavy_hex",
    "N": 10,
    "p": 2,
    "h_range": (1.0, 3.5),  # (h_min, h_max) you want to explore
}
# ══════════════════════════════════════════════════════════════════════════


def preflight_check(config: dict) -> dict:
    """Run pre-flight viability diagnostics for a pipeline configuration."""
    model = config["model"]
    topology = config["topology"]
    N = config["N"]
    p = config["p"]
    h_lo, h_hi = config["h_range"]
    
    report = {"config": config, "issues": [], "warnings": []}
    
    # 1. Model compatibility
    spec = get_model_spec(model)
    n_params = spec.total_params_for_p(p)
    report["n_params"] = n_params
    
    if model in ("heisenberg", "xy", "kitaev"):
        report["issues"].append(
            f"❌ Model '{model}' is incompatible with HVA+|+⟩^N. "
            f"The circuit cannot express the ground state at any h."
        )
        report["viable"] = False
        return report
    
    # 2. Expressibility boundary
    h_boundary = h_min_frontier(N, p, model, topology)
    report["h_min_boundary"] = h_boundary
    
    if h_lo < h_boundary:
        n_invalid = sum(1 for h in np.linspace(h_lo, h_hi, 20) if h < h_boundary)
        report["warnings"].append(
            f"⚠️  {n_invalid}/20 h-points fall below expressibility boundary "
            f"(h_min={h_boundary:.2f}). These will show ΔE/gap > 5%."
        )
    
    # 3. CX gate count
    lattice = make_lattice(topology, N, J=1.0, h=2.0)
    n_edges = len(lattice.edges)
    cx_count = 2 * n_edges * p  # Each RZZ decomposes to 2 CX
    report["cx_count"] = cx_count
    report["n_edges"] = n_edges
    
    if cx_count > 50:
        report["warnings"].append(
            f"⚠️  High CX count ({cx_count}). ZNE error mitigation may be "
            f"impractical on hardware (budget ~18-30 CX for reliable ZNE)."
        )
    
    # 4. COBYLA threshold
    if n_params > 8:
        report["warnings"].append(
            f"⚠️  High parameter count ({n_params}). VQE will auto-switch to "
            f"COBYLA (gradient-free), which may need more iterations."
        )
    
    # 5. Valid h-range
    valid_lo = max(h_lo, h_boundary)
    valid_hi = h_hi
    valid_span = valid_hi - valid_lo
    report["valid_h_range"] = (valid_lo, valid_hi)
    report["valid_span"] = valid_span
    
    if valid_span < 0.5:
        report["issues"].append(
            f"❌ Valid h-range too narrow ({valid_span:.2f}). "
            f"Need ≥ 0.5 span for meaningful MPNN training."
        )
    
    report["viable"] = len(report["issues"]) == 0
    return report


# Run the check
report = preflight_check(CONFIG)

# Visualize the lattice for this configuration
viz_lattice = make_lattice(CONFIG['topology'], CONFIG['N'], J=1.0, h=2.0)
fig = draw_lattice(CONFIG['topology'], CONFIG['N'], viz_lattice.edges,
                   h_value=sum(CONFIG['h_range'])/2,
                   title=f"Your System: {CONFIG['model']} on {CONFIG['topology']} (N={CONFIG['N']})")
plt.show()

# Show the HVA circuit
spec_viz = get_model_spec(CONFIG['model'])
qc_viz, _ = spec_viz.create_circuit(CONFIG['N'], CONFIG['p'], viz_lattice, **spec_viz.circuit_kwargs)
fig = draw_circuit(qc_viz, title=f"HVA Circuit: {CONFIG['model']} (N={CONFIG['N']}, p={CONFIG['p']})")
plt.show()

print("═" * 60)
print("  PRE-FLIGHT VIABILITY REPORT")
print("═" * 60)
print(f"  Model:    {CONFIG['model']}")
print(f"  Topology: {CONFIG['topology']}")
print(f"  N={CONFIG['N']}, p={CONFIG['p']}")
print(f"  h-range:  [{CONFIG['h_range'][0]}, {CONFIG['h_range'][1]}]")
print("─" * 60)
print(f"  Parameters:    {report['n_params']}")
print(f"  CX gates:      {report.get('cx_count', 'N/A')}")
print(f"  h_min boundary: {report.get('h_min_boundary', 'N/A'):.2f}")
print(f"  Valid range:   [{report.get('valid_h_range', ('?','?'))[0]:.2f}, "
      f"{report.get('valid_h_range', ('?','?'))[1]:.2f}]")
print("─" * 60)

if report["viable"]:
    print("  ✅ VIABLE — Pipeline should work for this configuration.")
else:
    print("  ❌ NOT VIABLE — See issues below.")

for issue in report.get("issues", []):
    print(f"  {issue}")
for warning in report.get("warnings", []):
    print(f"  {warning}")
print("═" * 60)

## 3. Topology Comparison — h_min vs Connectivity

Higher coordination number (more edges per site) makes the ground state
harder to express with shallow circuits.

In [ ]:
topologies = ["chain_1d", "heavy_hex", "ladder", "square", "triangular"]
p_fixed = 2
N_fixed = 10

fig, ax = plt.subplots(figsize=(8, 5))

h_mins = []
z_maxes = []
for topo in topologies:
    lattice = make_lattice(topo, N_fixed, J=1.0, h=2.0)
    n_edges = len(lattice.edges)
    z_max = max(sum(1 for (a, b) in lattice.edges if a == i or b == i) for i in range(N_fixed))
    h_min_val = h_min_frontier(N_fixed, p_fixed, "tfim", topo)
    h_mins.append(h_min_val)
    z_maxes.append(z_max)
    print(f"  {topo:12s}: z_max={z_max}, edges={n_edges}, h_min={h_min_val:.2f}")

ax.barh(topologies, h_mins, color=['#2ecc71', '#27ae60', '#f39c12', '#e74c3c', '#8e44ad'])
ax.axvline(1.0, color='red', ls='--', label='h_c = 1.0')
ax.set_xlabel('h_min (expressibility boundary)', fontsize=12)
ax.set_title(f'Expressibility Boundary by Topology (N={N_fixed}, p={p_fixed})', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Model Compatibility Matrix

Not all spin models work with the HVA ansatz. Here's the summary:

In [ ]:
print("\n┌─────────────────────────┬──────────┬───────────────────────────────────────┐")
print("│ Model                   │ Viable?  │ Notes                                 │")
print("├─────────────────────────┼──────────┼───────────────────────────────────────┤")
models_info = [
    ("tfim",                "✅ YES", "Best supported. 100% pass at p≥2"),
    ("tfim_longitudinal",   "✅ YES", "Same as TFIM (RZ adds zero CX cost)"),
    ("tfim_frustrated",     "✅ YES", "Noiseless only (27 CX at N=6)"),
    ("tfim_bond_resolved",  "✅ YES", "Crosses h_c! Best expressibility"),
    ("heisenberg",          "❌ NO",  "HVA+|+⟩^N incompatible (h>3.5 only)"),
    ("heisenberg_transverse","⚠️ PARTIAL", "Works only in paramagnetic (h>3.5)"),
    ("xy",                  "❌ NO",  "Same limitation as Heisenberg"),
    ("kitaev",              "❌ NO",  "XX+YY structure incompatible"),
]
for name, viable, notes in models_info:
    print(f"│ {name:<23s} │ {viable:<8s} │ {notes:<37s} │")
print("└─────────────────────────┴──────────┴───────────────────────────────────────┘")
print("\n💡 Tip: For Heisenberg, you need p ∝ N layers (Sumeet 2025).")
print("   The shallow HVA (p≤4) cannot capture the antiferromagnetic ground state.")